# Capas semánticas: ontologías y grafos

**Lección 3 · Clase 5.4** — el LLM más capaz del mundo no sabe qué significa `mnt`. Los datos de una empresa viven en esquemas crípticos, con abreviaturas que solo el equipo entiende, joins que nadie declaró y trampas silenciosas (¿esa columna está en pesos o en dólares?). Conectar un LLM *directamente* a esa base es pedirle que adivine.

Una **capa semántica** es la respuesta: una representación explícita de lo que los datos *significan* — qué es cada tabla, qué mide cada columna, en qué moneda, cómo se juntan — separada de los datos mismos. Y un **grafo** es su forma natural, porque el significado es relaciones: *esta columna mide ingresos*, *esta tabla junta con aquella*, *estas dos tablas hablan de lo mismo*.

En esta lección construimos una, por capas, sobre una base mockeada de un centro de esquí:

| | |
|---|---|
| **El problema** | Text-to-SQL a ciegas sobre nombres crípticos: el modelo adivina, y adivina mal. |
| **La base heredada** | 11 tablas SQLite con el desorden de la vida real (y una trampa de moneda escondida). |
| **Capa técnica** | El esquema como grafo: [neocarta](https://github.com/neo4j-labs/neocarta) (Neo4j Labs) ingiere la metadata a Neo4j. |
| **Capa de negocio** | La ontología: dominios, métricas, monedas y los joins que el esquema no declara. |
| **El recorrido** | Las preguntas que el grafo responde y el error que previene. |
| **Enriquecimiento LLM** | `gpt-5-mini` escribe descripciones y sinónimos en español, como capa aparte. |

En la lección 4 un agente usa este grafo para escribir SQL correcto — esta lección construye el mapa; la siguiente lo pone a manejar.

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q neocarta==0.8.0 neo4j==6.2.0 pandas==2.3.3 pyyaml==6.0.3 \
#   openai==2.53.0 matplotlib==3.11.1 networkx==3.6.1 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # el .env de la lección gana sobre variables heredadas del entorno

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USERNAME", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

# ¿Hay un Neo4j al que conectarse? (ver README: docker local o Aura Free)
from neo4j import GraphDatabase


def hay_neo4j() -> bool:
    try:
        with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, connection_timeout=3.0) as driver:
            driver.verify_connectivity()
        return True
    except Exception:
        return False


HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_NEO4J = hay_neo4j()
print("OPENAI_API_KEY presente:", HAY_OPENAI)
print(f"Neo4j accesible en {NEO4J_URI}:", HAY_NEO4J)
if not HAY_NEO4J:
    print("⚠️ Sin Neo4j las celdas de grafo se saltan (la base SQLite y la metadata corren igual).")
    print("   Local:  docker run -d --name neo4j-clase54 -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5")
    print("   Colab:  crea una instancia gratis en console.neo4j.io y pon NEO4J_URI/USERNAME/PASSWORD en secrets.")
if not HAY_OPENAI:
    print("⚠️ Sin OPENAI_API_KEY se saltan la demo del problema y el enriquecimiento LLM.")

# En Colab los .py y la ontología no existen: bajarlos del repo público.
import urllib.request
from pathlib import Path

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_4_integraciones/leccion3_capa_semantica/"
)


def asegurar(nombre: str) -> Path:
    ruta = Path(nombre)
    if not ruta.exists():
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


for archivo in ("crear_base_datos.py", "generar_metadata_csv.py", "construir_capa_negocio.py", "ontologia.yaml"):
    asegurar(archivo)

Path("outputs").mkdir(exist_ok=True)

## El problema, en una pregunta

Antes de construir nada, midamos qué pasa sin capa semántica. Le damos al modelo lo único que un esquema SQL sabe decir de sí mismo — nombres de tablas y columnas — y le hacemos la pregunta más básica de un negocio: *¿dónde están los ingresos?*

In [ ]:
import sqlite3

import crear_base_datos

crear_base_datos.crear(verboso=False)
conexion = sqlite3.connect("outputs/montania.db")

esquema_pelado = "\n".join(
    f"{tabla}({', '.join(col[1] for col in conexion.execute(f'PRAGMA table_info({tabla})'))})"
    for (tabla,) in conexion.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
)
print(esquema_pelado)

In [ ]:
if HAY_OPENAI:
    from openai import OpenAI

    cliente_openai = OpenAI()
    respuesta = cliente_openai.chat.completions.create(
        model="gpt-5-mini",
        messages=[{
            "role": "user",
            "content": (
                "Este es el esquema completo de la base de datos de un centro de esquí:\n\n"
                + esquema_pelado
                + "\n\nPregunta: ¿dónde están los ingresos totales del negocio y cómo los calculo? "
                "Responde en 4 líneas máximo, con el SQL."
            ),
        }],
    )
    print(respuesta.choices[0].message.content)
else:
    print("⛔ Falta OPENAI_API_KEY (la demo del problema se salta; el resto de la lección sigue).")

Mira la respuesta con calma, porque el error es silencioso. Hay dos salidas típicas y **las dos están mal**: o el modelo suma solo `trx_pos.mnt` (y deja fuera las reservas de agencias — una parte grande del negocio), o descubre que `res_agt.imp` parece plata y **la suma directo**. Lo que no puede hacer de ninguna manera es saber que `imp` está en **pesos argentinos**: los montos son del mismo orden de magnitud que los chilenos, y la moneda no aparece en ningún nombre, ningún tipo, ninguna FK. Es conocimiento tribal — y el número que sale de sumar pesos con pesos... equivocados, no existe.

Eso es exactamente lo que una capa semántica escribe en un lugar consultable.

## Ontologías, en una diapositiva

Una **ontología** es el vocabulario formal de un dominio: qué conceptos existen (*métrica*, *dominio*, *moneda*), qué relaciones son válidas (*una columna mide una métrica*), y qué instancias hay (*`res_agt.imp` mide `ingresos` en `USD`*). Hay dos grandes tradiciones para representarlas:

| | **RDF / OWL** (web semántica) | **Property graph** (Neo4j) |
|---|---|---|
| Unidad | Tripletas sujeto–predicado–objeto | Nodos y aristas con propiedades |
| Esquema | Formal y estricto (OWL, razonadores) | Flexible, se agrega sobre la marcha |
| Consulta | SPARQL | Cypher |
| Fortaleza | Interoperabilidad, inferencia lógica | Ergonomía, recorridos, ecosistema |

Usamos property graphs porque para *este* problema —darle contexto de negocio a un LLM— la ergonomía gana: Cypher se lee casi como lenguaje natural y el grafo crece por capas sin ceremonia. (En la frontera del final: OSI, un estándar de intercambio que apunta a que ambos mundos conversen.)

## La base que nos tocó heredar

Ya la creamos hace dos celdas ([`crear_base_datos.py`](crear_base_datos.py), determinística por semilla). Es el sistema operacional de un centro de esquí durante una temporada: ventas de punto de venta, reservas de agencias, mediciones de nieve, accesos a andariveles. Miremos el desorden de cerca:

In [ ]:
import pandas as pd

print("Una transacción de venta (trx_pos):")
display(pd.read_sql("SELECT * FROM trx_pos LIMIT 3", conexion))

print("Una reserva de agencia (res_agt) — imp parece un monto normal... ¿en qué moneda? El esquema no lo dice:")
display(pd.read_sql("SELECT * FROM res_agt LIMIT 3", conexion))

fks = pd.read_sql(
    """SELECT m.name AS tabla, f."from" AS columna, f."table" AS referencia
       FROM sqlite_master m JOIN pragma_foreign_key_list(m.name) f
       WHERE m.type = 'table'""",
    conexion,
)
print(f"Foreign keys declaradas en TODO el esquema: {len(fks)}")
display(fks)

Dos foreign keys declaradas... y siete joins más que el equipo usa todos los días (`canal_id` junta con `can_venta`, `sector_id` con `sec_mont`...) que **no están escritos en ninguna parte**. Más la trampa: `trx_pos.mnt` está en pesos **chilenos** y `res_agt.imp` en pesos **argentinos** — montos parecidos, monedas distintas — y ni el nombre, ni el tipo, ni la misteriosa `tc_d(f, vlr)` lo explican.

## Capa técnica: el esquema como grafo

Primera capa: poner en Neo4j lo que el esquema *sí* sabe de sí mismo. [neocarta](https://github.com/neo4j-labs/neocarta) (Neo4j Labs) hace exactamente esto — construye grafos de metadata — con conectores para BigQuery, Dataplex y **CSVs normalizados**, que es el camino para cualquier base que no tenga conector directo, como nuestra SQLite:

```
SQLite ──PRAGMA──▶ CSVs de metadata ──CSVConnector──▶ (:Database)-[:HAS_SCHEMA]→(:Schema)
                                                        -[:HAS_TABLE]→(:Table)-[:HAS_COLUMN]→(:Column)
```

[`generar_metadata_csv.py`](generar_metadata_csv.py) introspecta la base (`PRAGMA table_info`, `PRAGMA foreign_key_list`) y escribe los CSVs; el conector los ingiere. Fíjate qué viaja: tablas, columnas, tipos, las **2** FKs reales, valores de columnas catálogo, y el glosario de términos de negocio. Ninguna descripción — el esquema no las tiene.

In [ ]:
import generar_metadata_csv

_ = generar_metadata_csv.generar()

In [ ]:
if HAY_NEO4J:
    from neocarta.connectors.csv import CSVConnector

    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    conector = CSVConnector(
        csv_directory="outputs/metadata_csv",
        neo4j_driver=driver,
        database_name=NEO4J_DB,
    )
    conector.ingest()


    def cypher(consulta: str, **params) -> pd.DataFrame:
        """Ejecuta Cypher y devuelve un DataFrame."""
        with driver.session(database=NEO4J_DB) as sesion:
            return pd.DataFrame([r.data() for r in sesion.run(consulta, **params)])


    print("Censo del grafo tras la capa técnica:")
    display(cypher("""
        MATCH (n) WHERE NOT n:__neocarta_graph__ AND NOT n:__clase_5_4__
        RETURN labels(n)[0] AS etiqueta, count(*) AS nodos ORDER BY nodos DESC
    """))
else:
    print("⛔ Sin Neo4j: desde acá las celdas de grafo se saltan.")

## Capa de negocio: la ontología

Segunda capa: lo que **no** está en el esquema. [`ontologia.yaml`](ontologia.yaml) es el conocimiento del equipo puesto por escrito — léelo, es corto y es el corazón de la lección. Declara dominios de negocio, qué métrica mide cada columna, **en qué moneda**, de qué trata cada tabla, y los siete joins por convención.

[`construir_capa_negocio.py`](construir_capa_negocio.py) lo lee y escribe la capa sobre el grafo. Detalle que importa: cada arista lleva `source: 'ontologia'` (o `'convencion'`), así la capa se puede reconstruir sin tocar lo que cargó neocarta ni lo que agregue el LLM después — **las capas tienen procedencia y se administran por separado**.

In [ ]:
if HAY_NEO4J:
    import construir_capa_negocio

    construir_capa_negocio.construir()

## El recorrido: qué responde el grafo

**1. El mapa por dominio.** La pregunta de orientación que un esquema pelado no responde: ¿qué tablas hablan de qué?

In [ ]:
if HAY_NEO4J:
    display(cypher("""
        MATCH (t:Table)-[:IN_DOMAIN]->(d:BusinessDomain)
        RETURN d.name AS dominio, collect(t.name) AS tablas
        ORDER BY dominio
    """))

**2. Una métrica a través de las tablas.** `ABOUT` es la columna vertebral de la alineación: conecta tablas distintas que hablan de lo mismo. ¿Dónde viven los ingresos? — la pregunta con la que el modelo pelado falló:

In [ ]:
if HAY_NEO4J:
    display(cypher("""
        MATCH (t:Table)-[:ABOUT]->(m:Metric {name: 'ingresos'})
        RETURN t.name AS tabla, t.descripcion AS que_es
    """))

**3. La trampa que el grafo desarma.** Las dos tablas de ingresos, con la moneda de cada columna explícita — la respuesta que evita sumar peras con manzanas:

In [ ]:
if HAY_NEO4J:
    display(cypher("""
        MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)-[:MEASURES]->(:Metric {name: 'ingresos'})
        MATCH (c)-[:IN_CURRENCY]->(cur:Currency)
        RETURN t.name AS tabla, c.name AS columna, cur.code AS moneda
    """))

**4. Los joins para text-to-SQL.** Todas las formas válidas de juntar tablas — las 2 FKs reales y las 7 convenciones, cada una con su condición de join lista para pegar en un SQL:

In [ ]:
if HAY_NEO4J:
    display(cypher("""
        MATCH (:Table)-[:HAS_COLUMN]->(c1:Column)-[r:REFERENCES]->(c2:Column)
        RETURN r.criteria AS join_valido,
               CASE WHEN r.source = 'convencion' THEN 'convención del equipo' ELSE 'FK declarada' END AS origen
        ORDER BY origen, join_valido
    """))

**5. El glosario.** Los términos que usa la gente (*ticket promedio*, *forfait*, *temporada*) apuntando a su definición operativa — el puente entre cómo se habla y cómo se consulta:

In [ ]:
if HAY_NEO4J:
    display(cypher("""
        MATCH (g:Glossary)-[:HAS_CATEGORY]->(cat:Category)-[:HAS_BUSINESS_TERM]->(bt:BusinessTerm)
        RETURN cat.name AS categoria, bt.name AS termino, bt.description AS definicion
        ORDER BY categoria, termino
    """))

**6. El grafo, dibujado.** Tablas coloreadas por dominio, conectadas a las métricas de las que tratan y a las monedas en que miden — la capa de significado completa en una imagen:

In [ ]:
if HAY_NEO4J:
    import matplotlib.pyplot as plt
    import networkx as nx

    aristas = cypher("""
        MATCH (t:Table)-[r:ABOUT|IN_DOMAIN]->(concepto)
        RETURN t.name AS origen, type(r) AS tipo,
               coalesce(concepto.name, concepto.code) AS destino, t.dominio AS dominio
        UNION
        MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)-[r:IN_CURRENCY]->(cur:Currency)
        RETURN t.name AS origen, type(r) AS tipo, cur.code AS destino, t.dominio AS dominio
    """)

    G = nx.Graph()
    dominio_de = {}
    for _, fila in aristas.iterrows():
        G.add_edge(fila["origen"], fila["destino"])
        dominio_de[fila["origen"]] = fila["dominio"]

    colores_dominio = {"ventas": "#2a9d8f", "montania": "#457b9d", "clientes": "#e9c46a"}
    colores = [
        colores_dominio.get(dominio_de.get(nodo), "#e76f51" if nodo in dominio_de else "#adb5bd")
        for nodo in G.nodes
    ]
    tamanos = [1600 if nodo in dominio_de else 900 for nodo in G.nodes]

    plt.figure(figsize=(12, 7))
    posiciones = nx.spring_layout(G, seed=54, k=0.9)
    nx.draw_networkx(G, posiciones, node_color=colores, node_size=tamanos,
                     font_size=8, edge_color="#ced4da")
    plt.title("La capa semántica: tablas (por dominio) → métricas y monedas")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("outputs/grafo_conceptos.png", dpi=110)
    plt.show()

## Tercera capa: enriquecimiento con LLM

Las capas técnica y de negocio salieron de la introspección y de la ontología escrita a mano. La tercera la escribe un modelo: descripciones y **sinónimos en español** por tabla, para que después se pueda buscar como habla la gente ("facturación", "boletas", "clima") y no solo con los nombres crípticos.

Es la misma idea de procedencia: estas propiedades llevan el prefijo `llm_`, así se distinguen de lo curado a mano y se pueden regenerar (o borrar) sin tocar el resto.

In [ ]:
if HAY_NEO4J and HAY_OPENAI:
    import json as json_lib

    tablas_grafo = cypher("""
        MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
        RETURN t.name AS tabla, t.descripcion AS descripcion, collect(c.name) AS columnas
    """)

    filas_llm = []
    for _, fila in tablas_grafo.iterrows():
        respuesta = cliente_openai.chat.completions.create(
            model="gpt-5-mini",
            messages=[{
                "role": "user",
                "content": (
                    f"Tabla de un centro de esquí: {fila['tabla']}. "
                    f"Descripción: {fila['descripcion']}. Columnas: {', '.join(fila['columnas'])}.\n"
                    "Devuelve SOLO un JSON con: 'sinonimos' (lista de 4 términos en español, minúsculas, "
                    "que una persona de negocio usaría para referirse a estos datos)."
                ),
            }],
            response_format={"type": "json_object"},
        )
        sinonimos = json_lib.loads(respuesta.choices[0].message.content)["sinonimos"]
        filas_llm.append({"tabla": fila["tabla"], "sinonimos": sinonimos})
        print(f"  {fila['tabla']:<10} → {', '.join(sinonimos)}")

    with driver.session(database=NEO4J_DB) as sesion:
        sesion.run(
            """UNWIND $filas AS fila
               MATCH (t:Table {name: fila.tabla})
               SET t.llm_sinonimos = fila.sinonimos""",
            filas=filas_llm,
        ).consume()
    print("\n✓ sinónimos guardados como propiedad llm_sinonimos")
else:
    print("⛔ Esta sección necesita Neo4j Y OPENAI_API_KEY.")

Y ahora la búsqueda funciona en el idioma del negocio, no en el del esquema:

In [ ]:
if HAY_NEO4J and HAY_OPENAI:
    for palabra in ("venta", "nieve", "agencia"):
        resultado = cypher("""
            MATCH (t:Table)
            WHERE any(s IN coalesce(t.llm_sinonimos, []) WHERE toLower(s) CONTAINS $palabra)
               OR toLower(coalesce(t.descripcion, '')) CONTAINS $palabra
            RETURN t.name AS tabla
        """, palabra=palabra)
        print(f"  '{palabra}' → {sorted(resultado['tabla'].tolist()) if not resultado.empty else '(sin resultados)'}")

## La frontera

- **Búsqueda semántica de verdad**: neocarta trae embeddings y búsqueda vectorial/híbrida sobre la metadata (`neocarta[mcp]` expone todo como servidor MCP — la lección 1 y esta se juntan ahí). Nuestro `CONTAINS` es la versión de juguete.
- **Conectores reales**: lo que acá hicimos con CSVs, neocarta lo hace directo contra BigQuery y Dataplex; y **OSI** (Open Semantic Interchange) es el estándar emergente para intercambiar capas semánticas entre herramientas.
- **GraphRAG**: usar el grafo como contexto de recuperación va más allá de metadata — hay toda una familia de técnicas sobre grafos de conocimiento de documentos.
- **Gobernanza**: en una empresa real la ontología tiene dueños, versiones y proceso de cambio — es un producto de datos, no un script.

El grafo queda construido y esperando. **En la lección 4, un agente lo usa**: primero consulta la capa semántica para *entender*, y recién después escribe SQL para *calcular* — y medimos cuánto mejor le va que al agente que adivina.

In [ ]:
if HAY_NEO4J:
    driver.close()
conexion.close()
print("Listo. El grafo queda en Neo4j para la lección 4.")